In [1]:
## imports and functions for exploring the PAM Rodeo glider data

import pandas as pd
import numpy as np
import scipy.stats as stats
import matplotlib.pyplot as plt
import h5py
import os

import config
from noise_utils import inspect_h5_structure, build_noise_df, compute_depth_mask_flag

In [2]:
glider_id = config.GLIDER_ID

print(f'Analyzing {glider_id} (environment: {config.ENVIRONMENT})')
print(config.GLIDER_SCI_PATH)
print(config.PAM_NOISE_PATH)
print(config.PHASE_KEY_PATH)

print('###############################################################\n')
print('Reading glider science data from:', config.GLIDER_SCI_PATH)
glider_sci_df = pd.read_csv(config.GLIDER_SCI_PATH)
# 'time' is Unix epoch seconds (UTC) - confirmed against the PAM DateTime
# start (2026-01-28 22:23:58) and phase key deployment start (2026-01-28 22:16:00)
glider_sci_df['DateTime'] = pd.to_datetime(glider_sci_df['time'], unit='s')
print(glider_sci_df.head())
print('###############################################################\n')

print('Reading PAM noise data from:', config.PAM_NOISE_PATH)
print('Inspecting PAM h5 structure:')
pam_noise = h5py.File(config.PAM_NOISE_PATH, 'r')
inspect_h5_structure(pam_noise['GliderRodeo'])
pam_noise_df = build_noise_df(pam_noise)
print(pam_noise_df.head())
print('###############################################################\n')

print('Reading phase key data from:', config.PHASE_KEY_PATH)
phase_key_df = pd.read_excel(config.PHASE_KEY_PATH, sheet_name=glider_id.upper(), skiprows=1)
print(phase_key_df.head())
print('\n')

Analyzing sg607 (environment: local)
.\sg607_20260128_science_timeseries.csv
.\sg607_20260128.h5
./glider_rodeo_phase_key.xlsx
###############################################################

Reading glider science data from: .\sg607_20260128_science_timeseries.csv
   dive        time   latitude   longitude     depth  temperature  salinity  \
0     1  1769638886  21.212667 -158.157017  1.180368    25.500914       NaN   
1     1  1769638891  21.212667 -158.157017  1.170365    25.501297       NaN   
2     1  1769638896  21.212667 -158.157017  1.350420    25.479983       NaN   
3     1  1769638901  21.212667 -158.157017  1.230383    25.480549       NaN   
4     1  1769638906  21.212667 -158.157017  1.240386    25.470384       NaN   

   density  soundVelocity            DateTime  
0      NaN            NaN 2026-01-28 22:21:26  
1      NaN            NaN 2026-01-28 22:21:31  
2      NaN            NaN 2026-01-28 22:21:36  
3      NaN            NaN 2026-01-28 22:21:41  
4      NaN         

In [3]:
# PAM noise columns to bring onto glider_sci_df - add more here as needed (must be column names in pam_noise_df)
noise_vars_to_merge = ['broadband']
print(f'Noise columns to merge: {noise_vars_to_merge}')
# max allowed gap between a science row's timestamp and the nearest PAM sample -
# beyond this, the merged noise values are NaN instead of silently reusing a
# distant/stale PAM value (e.g. before PAM logging started, or during a gap)
noise_merge_tolerance = pd.Timedelta(config.NOISE_MERGE_TOLERANCE)
print(f'Merging PAM noise columns {noise_vars_to_merge} onto glider science data with tolerance {noise_merge_tolerance}...')

# drop first so re-running this cell (without restarting the kernel) doesn't collide with
# the previous run's merged columns and get silently renamed to *_x/*_y
glider_sci_df = glider_sci_df.drop(columns=noise_vars_to_merge, errors='ignore')

noise_subset = pam_noise_df[['DateTime'] + noise_vars_to_merge].sort_values('DateTime')
glider_sci_df = glider_sci_df.sort_values('DateTime')

glider_sci_df = pd.merge_asof(
    glider_sci_df,
    noise_subset,
    on='DateTime',
    direction='nearest',
    tolerance=noise_merge_tolerance
)

glider_sci_df.head()

Noise columns to merge: ['broadband']
Merging PAM noise columns ['broadband'] onto glider science data with tolerance 0 days 00:01:30...


,dive,time,latitude,longitude,depth,temperature,salinity,density,soundVelocity,DateTime,broadband
0,1,1769638886,21.212667,-158.157017,1.180368,25.500914,NaN,NaN,NaN,2026-01-28 22:21:26,NaN
1,1,1769638891,21.212667,-158.157017,1.170365,25.501297,NaN,NaN,NaN,2026-01-28 22:21:31,NaN
2,1,1769638896,21.212667,-158.157017,1.350420,25.479983,NaN,NaN,NaN,2026-01-28 22:21:36,NaN
3,1,1769638901,21.212667,-158.157017,1.230383,25.480549,NaN,NaN,NaN,2026-01-28 22:21:41,NaN
4,1,1769638906,21.212667,-158.157017,1.240386,25.470384,NaN,NaN,NaN,2026-01-28 22:21:46,NaN


In [4]:
# Assign a repeating 'phase' label to glider_sci_df based on which dive number
# range each phase_key_df row covers (Mode is the phase label, e.g. 'shakedown', 'fast')
glider_sci_df['phase'] = pd.NA

for _, row in phase_key_df.iterrows():
    start_dive = pd.to_numeric(row['Start Dive Number'], errors='coerce')
    stop_dive = pd.to_numeric(row['Stop Dive Number'], errors='coerce')

    if pd.isna(start_dive):
        continue  # no numeric start - can't place this phase on the dive axis (e.g. '-' for mini-rodeo)
    print(f'Processing phase: {row["Mode"]}, Dive range: {start_dive} to {stop_dive}')
    if pd.isna(stop_dive):
        # non-numeric stop (e.g. 'recovery' row's stop is 'WHICEAS', a location not a dive number) -
        # treat as open-ended through the last dive in the data
        stop_dive = glider_sci_df['dive'].max()

    mask = glider_sci_df['dive'].between(start_dive, stop_dive)
    glider_sci_df.loc[mask, 'phase'] = row['Mode']

glider_sci_df[['dive', 'phase']].drop_duplicates()

Processing phase: shakedown, Dive range: 1 to 11
Processing phase: fast, Dive range: 12 to 27
Processing phase: shallow dives, Dive range: 18 to 21
Processing phase: slow, Dive range: 28 to 39
Processing phase: intermediate, Dive range: 40 to 48
Processing phase: drift, Dive range: 49 to 58
Processing phase: mini-slow, Dive range: 59 to 60
Processing phase: mini-intermediate, Dive range: 61 to 62
Processing phase: mini-fast, Dive range: 63 to 64
Processing phase: recovery, Dive range: 65 to nan


,dive,phase
0,1,shakedown
127,2,shakedown
382,3,shakedown
741,4,shakedown
1164,5,shakedown
...,...,...
119090,61,mini-intermediate
121357,62,mini-intermediate
123663,63,mini-fast
125750,64,mini-fast


## QC

Toggled via `config.QC_APPLIED`. `depth_mask_flag` is always computed and added to
`glider_sci_df` (see `noise_utils.compute_depth_mask_flag`): for each dive/climb pair, the
bottom inflection is the max depth reached and the surface inflection is the min depth (e.g.
for depths `[0, 2, 3, 4, 5, 4, 3, 2, 1]` the bottom inflection is 5). Since a dive's own rows
can never exceed that same dive's own min/max, each dive's rows are checked against the
**previous** dive's inflection points instead - `depth_mask_flag = 1` if shallower than the
previous dive's surface inflection or deeper than its bottom inflection, else `0` (good range
of data). Known limitation: legitimate depth changes at a mission-phase transition (e.g.
shallow-dives -> slow/deep-dives) will trip this, since it's dive-to-dive, not mode-aware.

When `config.QC_APPLIED` is `True`, rows flagged by `depth_mask_flag` are dropped before
export. The toggle itself is recorded into `summary_meta.json` and shown on the presentation
deck's title slide either way, so it's always clear whether a given deck reflects QC'd data.

TODO: additional QC checks (NaN gaps, out-of-range CTD values, merge-tolerance misses) still
to be defined.

In [5]:
print(f'QC applied: {config.QC_APPLIED}')

glider_sci_df['depth_mask_flag'] = compute_depth_mask_flag(glider_sci_df)
n_flagged = int(glider_sci_df['depth_mask_flag'].sum())
print(f'{n_flagged:,} of {len(glider_sci_df):,} rows flagged by depth_mask_flag')

if config.QC_APPLIED:
    glider_sci_df = glider_sci_df[glider_sci_df['depth_mask_flag'] == 0].reset_index(drop=True)
    print(f'QC applied: dropped flagged rows, {len(glider_sci_df):,} rows remain')

QC applied: False
3,559 of 128,395 rows flagged by depth_mask_flag


## Export

- Everything is saved under a per-glider folder (`config.OUT_DIR`), which the stats/plots and
  presentation notebooks also read from/write to - so multiple gliders never collide or
  overwrite each other's outputs. Data (CSV/parquet) and figures (PNGs) go into their own
  `data/` and `figures/` subfolders (`config.DATA_DIR` / `config.FIGURES_DIR`) so they don't
  get mixed together; the final `.pptx` is saved directly in `config.OUT_DIR`.
- `glider_sci_df` (merged science + broadband SPL + phase) -> CSV for the stats/plots notebook.
- The full hybrid millidecade spectrum (2797 bands, `pam_noise_df`) -> parquet, since it's too
  large for a practical CSV (2797 cols x ~15876 rows). Only `DateTime` + the hybrid bands are
  exported here since that's the only spectrum resolution the stats/plots notebook uses.

In [6]:
# data (CSV/parquet) goes in its own subfolder (config.DATA_DIR), separate from the figures
# the stats/plots notebook will save into config.FIGURES_DIR - both are created on import by config.py
glider_export_path = os.path.join(config.DATA_DIR, f'{glider_id}_merged_science.csv')
glider_sci_df.to_csv(glider_export_path, index=False)
print(f'Saved merged science data to {glider_export_path} ({len(glider_sci_df):,} rows)')

hybrid_cols = [c for c in pam_noise_df.columns if c.startswith('hybridMiliDecLevels_')]
spectrum_export_path = os.path.join(config.DATA_DIR, f'{glider_id}_pam_spectrum.parquet')
pam_noise_df[['DateTime'] + hybrid_cols].to_parquet(spectrum_export_path, index=False)
print(f'Saved PAM hybrid millidecade spectrum to {spectrum_export_path} '
      f'({len(hybrid_cols)} bands x {len(pam_noise_df):,} samples)')

Saved merged science data to ./noise_analysis_outputs\sg607\data\sg607_merged_science.csv (128,395 rows)
Saved PAM hybrid millidecade spectrum to ./noise_analysis_outputs\sg607\data\sg607_pam_spectrum.parquet (2797 bands x 15,876 samples)
